# Global goSPL reconstruction at 50 Ma

This notebook adapts the export and global-map pieces in `gospl-global-workflows` to plot the archived reconstruction used by Salles et al. (2023), *Hundred million years of landscape dynamics from catchment to global scale*. It produces separate figures patterned after Fig. 1A (physiography and major rivers) and Fig. 1B (cumulative erosion and deposition).

The reproducible conda specification is stored beside this notebook in `gospl_global_environment.yml`. Create it with `conda env create -f gospl_global_environment.yml`, then run this notebook in the `gospl-global` environment.

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

DATA_DIRECTORY = Path("/mnt/lochz/5AEK_DATA/5aek4-osfstorage-archive")
PLOT_TIME_MA = 0
TIME_STEP_MYR = 5
OUTPUT_DIRECTORY = DATA_DIRECTORY/"img_10152026"
PLOT_STRIDE = 2  # Plot at 0.2 degrees while retaining the source data unchanged.
RIVER_DISCHARGE_THRESHOLD = 5.0e10  # m^3/yr

if type(PLOT_TIME_MA) is not int or not 0 <= PLOT_TIME_MA <= 90:
    raise ValueError("PLOT_TIME_MA must be an integer between 0 and 90")
if PLOT_TIME_MA % TIME_STEP_MYR != 0:
    raise ValueError(f"PLOT_TIME_MA must be divisible by {TIME_STEP_MYR}")

data_file = DATA_DIRECTORY / f"physiography-{PLOT_TIME_MA:02d}Ma.nc"
older_data_file = DATA_DIRECTORY / f"physiography-{PLOT_TIME_MA + TIME_STEP_MYR:02d}Ma.nc"
for archive_file in (data_file, older_data_file):
    if not archive_file.is_file():
        raise FileNotFoundError(f"Cannot find the goSPL archive file: {archive_file}")

dataset = xr.open_dataset(data_file)
older_dataset = xr.open_dataset(older_data_file)
required_fields = {"elevation", "erodep", "flowDischarge"}
missing_fields = required_fields.difference(dataset.data_vars)
if missing_fields:
    raise KeyError(f"Missing required fields: {sorted(missing_fields)}")
if "erodep" not in older_dataset.data_vars:
    raise KeyError(f"Missing required field 'erodep' in {older_data_file}")

# Drop the duplicated +180 degree column and thin only for rendering.
plot_data = dataset.isel(
    latitude=slice(None, None, PLOT_STRIDE),
    longitude=slice(0, -1, PLOT_STRIDE),
).load()
older_plot_data = older_dataset[["erodep"]].isel(
    latitude=slice(None, None, PLOT_STRIDE),
    longitude=slice(0, -1, PLOT_STRIDE),
).load()
plot_data[["elevation", "erodep", "flowDischarge"]]

In [ ]:
source_crs = ccrs.PlateCarree()
view_crs = ccrs.Mollweide(central_longitude=-20)
longitude, latitude = np.meshgrid(plot_data.longitude, plot_data.latitude)



def finish_global_axis(axis):
    axis.set_global()
    axis.gridlines(
        color="0.25", linewidth=0.35, alpha=0.35, linestyle="--"
    )
    axis.spines["geo"].set_edgecolor("0.15")
    axis.spines["geo"].set_linewidth(0.8)

## Figure 1A-style physiography and major rivers

Elevation and bathymetry use the archived `elevation` field. The reconstructed coastline is the zero-elevation contour. Major rivers are selected from `flowDischarge` using the threshold configured above.

### Create the same colorbar

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize

positions = np.array([
    9, 27, 46, 65, 84, 103, 122, 141,
    150, 157, 165, 174, 183, 197
])
positions = (positions - positions[0]) / (positions[-1] - positions[0])

rgb = np.array([
    [35, 27, 73],
    [45, 51, 125],
    [30, 77, 158],
    [38, 102, 174],
    [46, 128, 197],
    [50, 155, 217],
    [72, 180, 230],
    [81, 202, 254],
    [151, 201, 159],
    [206, 175, 96],
    [166, 91, 55],
    [130, 86, 90],
    [120, 108, 109],
    [251, 251, 251],
]) / 255.0

elevation_cmap = LinearSegmentedColormap.from_list(
    "reproduced", list(zip(positions, rgb)), N=256
)

river_cmap = LinearSegmentedColormap.from_list(
    "major_rivers", ["#4a9bd3", "#08306b"]
)

fig, ax = plt.subplots(figsize=(8, 1.2))
fig.subplots_adjust(bottom=0.55, top=0.85)

cb = fig.colorbar(
    plt.cm.ScalarMappable(
        norm=Normalize(vmin=-11, vmax=4),
        cmap=elevation_cmap,
    ),
    cax=ax,
    orientation="horizontal",
)

cb.set_ticks([-11, -8, -4, 0, 4])
cb.set_ticklabels(["−11", "−8", "−4", "0", "4"])
cb.set_ticks(np.arange(-10, 4, 2), minor=True)
cb.ax.tick_params(which="major", labelsize=14, length=5)
cb.ax.tick_params(which="minor", length=4)
cb.outline.set_linewidth(0.8)

plt.show()

In [ ]:
elevation = plot_data.elevation.values
discharge = plot_data.flowDischarge.values
major_rivers = np.ma.masked_less(discharge, RIVER_DISCHARGE_THRESHOLD)

figure_a = plt.figure(figsize=(9, 9), constrained_layout=True)
axis_a = figure_a.add_subplot(1, 1, 1, projection=view_crs)
topography = axis_a.pcolormesh(
    longitude,
    latitude,
    elevation/1e3,
    transform=source_crs,
    cmap=elevation_cmap,
    norm=Normalize(vmin=-11, vmax=4),
    shading="auto",
    rasterized=True,
)
axis_a.contour(
    longitude, latitude, elevation, levels=[0], colors="#292929",
    linewidths=0.45, transform=source_crs,
)
axis_a.pcolormesh(
    longitude,
    latitude,
    np.log10(major_rivers),
    transform=source_crs,
    cmap=river_cmap,
    vmin=np.log10(RIVER_DISCHARGE_THRESHOLD),
    vmax=np.log10(float(plot_data.flowDischarge.max())),
    shading="auto",
    alpha=0.9,
    rasterized=True,
)
finish_global_axis(axis_a)
axis_a.set_title(f"A  Global physiography and major rivers at {PLOT_TIME_MA} Ma", loc="left", weight="bold")
colorbar_a = figure_a.colorbar(topography, ax=axis_a, orientation="horizontal", shrink=0.62, pad=0.035)
colorbar_a.set_label("Elevation (m)")
figure_a.savefig(OUTPUT_DIRECTORY / f"gospl_figure1a_{PLOT_TIME_MA:02d}Ma.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 1B-style cumulative erosion and deposition

The archived `erodep` field is shown with a zero-centered diverging scale: negative values are net erosion and positive values are net deposition.

In [ ]:
erosion_deposition = plot_data.erodep.values
erodep_cmap = LinearSegmentedColormap.from_list(
    "erosion_deposition",
    [
        (0.00, "#253494"),
        (0.40, "#2c7fb8"),
        (0.45, "#7fcdbb"),
        (0.50, "#f7f7f7"),
        (0.55, "#fdae61"),
        (0.60, "#d73027"),
        (1.00, "#7f0000"),
    ],
)
erodep_norm = TwoSlopeNorm(vmin=-5000, vcenter=0, vmax=5000)

figure_b = plt.figure(figsize=(9, 9), constrained_layout=True)
axis_b = figure_b.add_subplot(1, 1, 1, projection=view_crs)
erodep_image = axis_b.pcolormesh(
    longitude,
    latitude,
    erosion_deposition,
    transform=source_crs,
    cmap=erodep_cmap,
    norm=erodep_norm,
    shading="auto",
    rasterized=True,
)
axis_b.contour(
    longitude, latitude, elevation, levels=[0], colors="#303030",
    linewidths=0.45, transform=source_crs,
)
finish_global_axis(axis_b)
axis_b.set_title(f"B  Cumulative erosion and deposition at {PLOT_TIME_MA} Ma", loc="left", weight="bold")
colorbar_b = figure_b.colorbar(erodep_image, ax=axis_b, orientation="horizontal", shrink=0.62, pad=0.035)
colorbar_b.set_ticks([-4000, -2000, 0, 2000, 4000])
colorbar_b.set_label("Erosion (-) / deposition (+) (m)")
figure_b.savefig(OUTPUT_DIRECTORY / f"gospl_figure1b_{PLOT_TIME_MA:02d}Ma.png", dpi=300, bbox_inches="tight")
plt.show()

## Erosion and sedimentation rate

The signed average rate is calculated from adjacent cumulative `erodep` snapshots. Negative values indicate erosion and positive values indicate sedimentation.

In [ ]:
erosion_sedimentation_rate = (
    plot_data.erodep.values - older_plot_data.erodep.values
) / TIME_STEP_MYR
rate_limit = np.nanpercentile(np.abs(erosion_sedimentation_rate), 99)
rate_norm = TwoSlopeNorm(vmin=-rate_limit, vcenter=0, vmax=rate_limit)


erodep_cmap_1 = LinearSegmentedColormap.from_list(
    "erosion_deposition",
    [
        (0.00, "#253494"),
        (0.25, "#2c7fb8"),
        (0.40, "#7fcdbb"),
        (0.50, "#f7f7f7"),
        (0.60, "#fdae61"),
        (0.75, "#d73027"),
        (1.00, "#7f0000"),
    ],
)

rate_figure = plt.figure(figsize=(9, 9), constrained_layout=True)
rate_axis = rate_figure.add_subplot(1, 1, 1, projection=view_crs)
rate_image = rate_axis.pcolormesh(
    longitude,
    latitude,
    erosion_sedimentation_rate,
    transform=source_crs,
    cmap=erodep_cmap_1,
    norm=rate_norm,
    shading="auto",
    rasterized=True,
)
rate_axis.contour(
    longitude, latitude, elevation, levels=[0], colors="#303030",
    linewidths=0.45, transform=source_crs,
)
finish_global_axis(rate_axis)
rate_axis.set_title(
    f"Average erosion and sedimentation rate, {PLOT_TIME_MA + TIME_STEP_MYR}--{PLOT_TIME_MA} Ma",
    loc="left",
    weight="bold",
)
rate_colorbar = rate_figure.colorbar(
    rate_image, ax=rate_axis, orientation="horizontal", shrink=0.62, pad=0.035
)
rate_colorbar.set_label("Erosion (-) / sedimentation (+) (m/Myr)")
rate_figure.savefig(
    OUTPUT_DIRECTORY / f"gospl_erosion_sedimentation_rate_{PLOT_TIME_MA:02d}Ma.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()